# DB 데이터 추출 및 전처리

PostgreSQL DB에서 Steam 인디게임 데이터를 추출하여 `data/raw/`에 원본 CSV로 저장하고,
이어서 각 데이터셋을 전처리하여 `data/preprocessed/`에 저장한다.

## DB 추출 대상

| 테이블 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_tags` | SteamSpy 태그 데이터 | `data/raw/steam_indie_tags.csv` |
| `steam_indie_reviews` | 리뷰 원문 및 작성자 정보 | `data/raw/steam_indie_reviews.csv` |
| `steam_indie_review_histogram` | 월별/일별 리뷰 집계 | `data/raw/steam_indie_review_histogram.csv` |
| `steam_indie_review_summary` | 게임별 리뷰 요약 통계 | `data/raw/steam_indie_review_summary.csv` |
| `steam_app_details` | 게임 상세 정보 (Indie 장르만) | `data/raw/steam_app_details.csv` |
| `steamspy_indie_games` | SteamSpy 수집 인디게임 전체 데이터 | `data/raw/steamspy_indie_games.csv` |

## 전처리 결과

| 대상 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_games` | 분석 모집단 필터링·정제·병합 결과 | `data/preprocessed/steam_indie_games.csv` |
| `steam_indie_review_histogram` | 파생 컬럼 추가 | `data/preprocessed/steam_indie_review_histogram.csv` |
| `steam_indie_reviews` | 결측 제거·날짜 변환·플레이타임 정제 | `data/preprocessed/steam_indie_reviews.csv` |

## 라이브러리 임포트 및 DB 연결

In [22]:
import sys
import ast
import json as _json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve().parents[1] / 'src'))

from utils.db import get_engine

warnings.filterwarnings('ignore')

RAW_DIR = Path('..').resolve().parents[1] / 'data' / 'raw'
PREPROCESSED_DIR = Path('..').resolve().parents[1] / 'data' / 'preprocessed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = get_engine()
print('DB 연결 성공')
print(f'RAW_DIR: {RAW_DIR}')
print(f'PREPROCESSED_DIR: {PREPROCESSED_DIR}')

DB 연결 성공
RAW_DIR: /Users/jin/Develop/codingclub/game-analysis/data/raw
PREPROCESSED_DIR: /Users/jin/Develop/codingclub/game-analysis/data/preprocessed


## 1. steam_indie_tags

SteamSpy API로 수집한 게임 태그 데이터. `tags` 컬럼은 JSONB 형식으로 저장되어 있어 문자열로 변환한다.

In [23]:
df_tags = pd.read_sql('SELECT * FROM steam_indie_tags ORDER BY appid', conn)

df_tags['tags'] = df_tags['tags'].apply(
    lambda x: _json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x
)

out = RAW_DIR / 'steam_indie_tags.csv'
df_tags.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_tags: {len(df_tags):,}행 → {out.name}')
print(f'컬럼: {df_tags.columns.tolist()}')
df_tags.head(3)

steam_indie_tags: 9,706행 → steam_indie_tags.csv
컬럼: ['appid', 'name', 'developer', 'publisher', 'owners', 'positive', 'negative', 'price', 'tags']


,appid,name,developer,publisher,owners,positive,negative,price,tags
0,226620,Desktop Dungeons,QCF Design,QCF Design,"200,000 .. 500,000",1912,364,1499,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
1,230210,ASYLUM,Senscape,Senscape,"0 .. 20,000",303,45,2499,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""..."
2,251570,7 Days to Die,The Fun Pimps,The Fun Pimps Entertainment LLC,"10,000,000 .. 20,000,000",327889,42157,4499,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""..."


## 2. steam_indie_reviews

수집된 전체 리뷰 데이터. 236,000건 이상으로 용량이 크므로 청크 단위로 읽어 저장한다.

In [24]:
out = RAW_DIR / 'steam_indie_reviews.csv'

chunk_size = 50000
total = 0
for i, chunk in enumerate(
    pd.read_sql('SELECT * FROM steam_indie_reviews ORDER BY appid, timestamp_created', conn, chunksize=chunk_size)
):
    chunk.to_csv(out, index=False, encoding='utf-8-sig', mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  청크 {i+1}: {total:,}행 저장 완료')

print(f'\nsteam_indie_reviews: 총 {total:,}행 → {out.name}')

  청크 1: 50,000행 저장 완료
  청크 2: 100,000행 저장 완료
  청크 3: 150,000행 저장 완료
  청크 4: 200,000행 저장 완료
  청크 5: 236,379행 저장 완료

steam_indie_reviews: 총 236,379행 → steam_indie_reviews.csv


## 3. steam_indie_review_histogram

게임별 월별(`rollups`) 및 일별(`recent`) 리뷰 집계 데이터.

In [25]:
df_hist = pd.read_sql(
    'SELECT * FROM steam_indie_review_histogram ORDER BY appid, data_type, date', conn
)

out = RAW_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_histogram: {len(df_hist):,}행 → {out.name}')
print(f'컬럼: {df_hist.columns.tolist()}')
print(f'\ndata_type 분포:')
print(df_hist['data_type'].value_counts().to_string())
df_hist.head(3)

steam_indie_review_histogram: 11,782행 → steam_indie_review_histogram.csv
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

data_type 분포:
data_type
recent     5976
rollups    5806


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent


## 4. steam_indie_review_summary

게임별 리뷰 요약 통계 (review_score, 긍정/부정 수 등). 수집 시점의 전체 누적 리뷰 기준이다.

In [26]:
df_summary = pd.read_sql('SELECT * FROM steam_indie_review_summary ORDER BY appid', conn)

out = RAW_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_summary: {len(df_summary):,}행 → {out.name}')
print(f'컬럼: {df_summary.columns.tolist()}')
df_summary.head(3)

steam_indie_review_summary: 200행 → steam_indie_review_summary.csv
컬럼: ['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews
0,402160,5,Mixed,152,205,357
1,437440,5,Mixed,66,30,96
2,444690,5,Mixed,111,135,246


## 5. steam_app_details

Steam Store API로 수집한 게임 상세 정보 (설명, 장르, 카테고리 등 전체 필드). `genres` 배열에 `Indie`가 포함된 게임만 조회한다.

In [27]:
df_app_details = pd.read_sql(
    """
    SELECT a.*
    FROM steam_app_details a
    INNER JOIN steamspy_indie_games s ON a.appid = s.appid
    ORDER BY a.appid
    """,
    conn
)

out = RAW_DIR / 'steam_app_details.csv'
df_app_details.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_app_details: {len(df_app_details):,}행 → {out.name}')
print(f'컬럼: {df_app_details.columns.tolist()}')
df_app_details.head(3)

steam_app_details: 58,453행 → steam_app_details.csv
컬럼: ['appid', 'name', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'developers', 'publishers', 'genres', 'categories', 'coming_soon', 'release_date', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


,appid,name,type,is_free,controller_support,short_description,supported_languages,developers,publishers,genres,...,final_formatted,windows,mac,linux,recommendations_total,metacritic_score,metacritic_url,achievements_total,header_image,website
0,1002,Rag Doll Kung Fu,game,False,NaN,A piece of Steam history - THE FIRST EVER NON ...,English,Mark Healey,Mark Healey,Indie,...,"₩ 1,100",True,False,False,NaN,69.0,https://www.metacritic.com/game/pc/rag-doll-ku...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.ragdollkungfu.com/
1,1500,Darwinia,game,False,full,"Darwinia blends real-time strategy, action, an...","English, German, French, Italian, Spanish - Spain",Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,773.0,84.0,https://www.metacritic.com/game/pc/darwinia?ft...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.darwinia.co.uk/
2,1510,Uplink,game,False,NaN,Uplink lets you play as a freelance hacker tac...,English,Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,1744.0,75.0,https://www.metacritic.com/game/pc/uplink-hack...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.uplink.co.uk/


## 6. steamspy_indie_games

SteamSpy API로 수집한 인디게임 전체 데이터.

In [28]:
df_steamspy = pd.read_sql('SELECT * FROM steamspy_indie_games ORDER BY appid', conn)

out = RAW_DIR / 'steamspy_indie_games.csv'
df_steamspy.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steamspy_indie_games: {len(df_steamspy):,}행 → {out.name}')
print(f'컬럼: {df_steamspy.columns.tolist()}')
df_steamspy.head(3)

steamspy_indie_games: 61,266행 → steamspy_indie_games.csv
컬럼: ['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1002,Rag Doll Kung Fu,"20,000 .. 50,000",91,30,99,0,Rag Doll Kung Fu,game,['Indie'],"12 Oct, 2005",Mark Healey
1,1500,Darwinia,"0 .. 20,000",864,216,1199,2,Darwinia,game,"['Indie', 'Strategy']","1 Dec, 2005",Introversion Software
2,1510,Uplink,"500,000 .. 1,000,000",2143,216,1199,2,Uplink,game,"['Indie', 'Strategy']","23 Aug, 2006",Introversion Software


## 7. DB 연결 종료 및 저장 결과 요약

In [29]:
conn.dispose()
print('DB 연결 종료')

print('\n=== 저장 완료 파일 목록 ===')
for f in sorted(RAW_DIR.glob('steam*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<50} {size_mb:>7.1f} MB')

DB 연결 종료

=== 저장 완료 파일 목록 ===
  steam_app_details.csv                                 38.4 MB
  steam_indie_review_histogram.csv                       1.1 MB
  steam_indie_review_summary.csv                         0.0 MB
  steam_indie_reviews.csv                               76.6 MB
  steam_indie_tags.csv                                   4.1 MB
  steamspy_indie_games.csv                               8.4 MB


---

## steam_indie_games 전처리

`steamspy_indie_games`를 기반으로 메인 분석 모집단을 선별하고,
`steam_indie_review_summary`, `steam_app_details`, `steam_indie_tags` 데이터를 병합하여
최종 분석 데이터셋을 구성한다.

### 데이터 로드 및 파생 컬럼 생성

분석에 필요한 파생 컬럼을 생성한다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `release_date`: datetime 변환
- `genres`: 문자열 → 리스트 파싱

In [30]:
df = df_steamspy.copy()

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres'] = df['genres'].apply(parse_genres)

print(f'원본: {len(df):,}개')
print(f'Early Access: {df["genres"].apply(lambda gl: "Early Access" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Early Access" in gl).mean():.1%})')
print(f'Free To Play: {df["genres"].apply(lambda gl: "Free To Play" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Free To Play" in gl).mean():.1%})')

원본: 61,266개
Early Access: 6,340개 (10.3%)
Free To Play: 3,446개 (5.6%)


### 분석 대상 필터링

다음 조건을 모두 만족하는 게임을 메인 분석 모집단으로 선별한다.

- 출시연도: 2023 ~ 2025년
- 리뷰 수: 10개 이상
- Early Access 제외
- Free to Play 제외

Early Access와 F2P 게임은 별도 데이터프레임(`df_ea`, `df_f2p`)으로 보존한다.

In [31]:
MIN_REVIEWS = 10

is_ea  = df['genres'].apply(lambda gl: 'Early Access' in gl)
is_f2p = df['genres'].apply(lambda gl: 'Free To Play' in gl)

df_ea  = df[is_ea].copy()
df_f2p = df[~is_ea & is_f2p].copy()
df_f   = df[
    (df['total_reviews'] >= MIN_REVIEWS) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year <= 2025) &
    (~is_ea) &
    (~is_f2p)
].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P 제외          : {len(df_f2p):,}개 → 별도 분석')
print(f'메인 모집단       : {len(df_f):,}개  (2023~2025년, 리뷰 {MIN_REVIEWS}개 이상, EA·F2P 제외)')
print(f'\n출시연도 분포 (메인):')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P 제외          : 3,064개 → 별도 분석
메인 모집단       : 9,692개  (2023~2025년, 리뷰 10개 이상, EA·F2P 제외)

출시연도 분포 (메인):
release_date
2023    3498
2024    4180
2025    2014


### 컬럼 정제

분석에 적합한 형태로 컬럼을 정리한다.

- `name`: `name_store` 우선, 없으면 `spy_name` 사용
- `price`: `price_spy` 컬럼명 변경
- `release_date`: `yyyy-MM-dd` 포맷으로 통일
- 불필요 컬럼(`spy_name`, `name_store`, `type`) 제거

In [32]:
df_f['name'] = df_f['name_store'].fillna(df_f['spy_name'])
df_f = df_f.rename(columns={'price_spy': 'price'})
df_f['release_date'] = df_f['release_date'].dt.strftime('%Y-%m-%d')
df_f = df_f.drop(columns=['spy_name', 'name_store', 'type'])

print('전처리 완료')
print(df_f[['name', 'release_date', 'price']].head())

전처리 완료
                                   name release_date  price
564                    Desktop Dungeons   2023-04-18   1499
598                              ASYLUM   2025-03-13   2499
848                       7 Days to Die   2024-07-25   4499
868   Defender's Quest 2: Mists of Ruin   2025-01-30   1999
1178                 Secrets of Grindea   2024-02-29   1499


### steam_indie_review_summary 병합

- `total_reviews`, `positive`, `negative` → `steam_indie_review_summary` 값으로 대체
- `review_score`, `review_score_desc` 컬럼 추가

In [33]:
common_cols_reviews = sorted(set(df_f.columns) & set(df_summary.columns))
print('games vs review_summary 공통 컬럼:', common_cols_reviews)

replace_cols = df_summary[['appid', 'total_reviews', 'total_positive', 'total_negative']]
extra_cols   = df_summary.drop(columns=['total_reviews', 'total_positive', 'total_negative'])

df_f = df_f.merge(replace_cols, on='appid', how='left', suffixes=('_old', ''))
df_f['total_reviews'] = df_f['total_reviews'].combine_first(df_f['total_reviews_old'])
df_f['positive']      = df_f['total_positive'].combine_first(df_f['positive'])
df_f['negative']      = df_f['total_negative'].combine_first(df_f['negative'])
df_f = df_f.drop(columns=['total_reviews_old', 'total_positive', 'total_negative'])

df_f = df_f.merge(extra_cols, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())

games vs review_summary 공통 컬럼: ['appid', 'total_reviews']
병합 결과 shape: (9692, 13)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'name', 'total_reviews', 'review_score', 'review_score_desc']


### steam_app_details 병합

- 공통 컬럼 (`name`, `developers`, `genres`, `release_date`) → `steam_app_details` 값으로 대체
- 나머지 `steam_app_details` 컬럼은 그대로 추가

In [34]:
common_cols_appdetails = sorted(set(df_f.columns) & set(df_app_details.columns))
print('games vs app_details 공통 컬럼:', common_cols_appdetails)

replace_cols2      = ['name', 'developers', 'genres', 'release_date']
appdetails_replace = df_app_details[['appid'] + replace_cols2]
appdetails_extra   = df_app_details.drop(columns=replace_cols2)

df_f = df_f.merge(appdetails_replace, on='appid', how='left', suffixes=('_old', ''))
for col in replace_cols2:
    df_f[col] = df_f[col].combine_first(df_f[f'{col}_old'])
    df_f = df_f.drop(columns=[f'{col}_old'])

df_f = df_f.merge(appdetails_extra, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())

games vs app_details 공통 컬럼: ['appid', 'developers', 'genres', 'name', 'release_date']
병합 결과 shape: (9692, 36)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'publishers', 'categories', 'coming_soon', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


### 타입 변환 및 컬럼 정리

- `price`: 정수(원 단위) → float (`/100`)
- `total_reviews`, `release_date`: numeric/datetime 변환
- `owners` 범위 문자열 → `owners_lower`(하한) + `owners_higher`(상한) 분리 후 `owners`, `ccu` 제거
- 분석에 불필요한 컬럼 15개 제거 (`type`, `is_free`, `controller_support`, `supported_languages`, `coming_soon`, `currency`, `initial`, `final`, `discount_percent`, `initial_formatted`, `final_formatted`, `metacritic_score`, `metacritic_url`, `header_image`, `website`)

In [35]:
df_f['price'] = pd.to_numeric(df_f['price'], errors='coerce') / 100
df_f['total_reviews'] = pd.to_numeric(df_f['total_reviews'], errors='coerce')
df_f['release_date'] = pd.to_datetime(df_f['release_date'], errors='coerce')

owners_clean = df_f['owners'].str.replace(',', '', regex=False)
df_f['owners_lower'] = pd.to_numeric(owners_clean.str.split(r'\.\.').str[0].str.strip(), errors='coerce')
df_f['owners_higher'] = pd.to_numeric(owners_clean.str.split(r'\.\.').str[1].str.strip(), errors='coerce')
df_f = df_f.drop(columns=['owners', 'ccu'])

# 분석에 불필요한 컬럼 제거
drop_cols = [
    'type', 'is_free', 'controller_support', 'supported_languages',
    'coming_soon', 'currency', 'initial', 'final', 'discount_percent',
    'initial_formatted', 'final_formatted', 'metacritic_score',
    'metacritic_url', 'header_image', 'website'
]
df_f = df_f.drop(columns=[c for c in drop_cols if c in df_f.columns])

print('타입 변환 완료')
print(df_f[['price', 'total_reviews', 'owners_lower', 'owners_higher', 'release_date']].dtypes)
print(f'\nshape: {df_f.shape}')
print(f'컬럼: {df_f.columns.tolist()}')

타입 변환 완료
price                   float64
total_reviews           float64
owners_lower              int64
owners_higher             int64
release_date     datetime64[us]
dtype: object

shape: (9692, 21)
컬럼: ['appid', 'positive', 'negative', 'price', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher']


### steam_indie_tags 정제 및 병합

`steam_indie_tags`에서 분석에 불필요한 컬럼을 제거하고, `df_f`와 inner join한다.

- 제거 컬럼: `owners`, `price`, `positive`, `negative`, `name`, `developer`, `publisher` (`steam_app_details` 기준 값이 이미 있음)
- 보존 컬럼: `appid`, `tags`
- `df_f`와 `appid` 기준 inner join → 분석 대상에 태그 정보가 없는 게임 제외

In [36]:
drop_cols = ['owners', 'price', 'positive', 'negative', 'name', 'developer', 'publisher']
df_tags_clean = df_tags.drop(columns=[col for col in drop_cols if col in df_tags.columns])

print(f'tags 정제 후 컬럼: {df_tags_clean.columns.tolist()}')
print(f'tags 행 수: {len(df_tags_clean):,}')

before = len(df_f)
df_f = df_f.merge(df_tags_clean, on='appid', how='inner')
print(f'\ninner join 결과: {before:,} → {len(df_f):,}행 ({before - len(df_f):,}개 제외)')
print(f'컬럼: {df_f.columns.tolist()}')

tags 정제 후 컬럼: ['appid', 'tags']
tags 행 수: 9,706

inner join 결과: 9,692 → 9,692행 (0개 제외)
컬럼: ['appid', 'positive', 'negative', 'price', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher', 'tags']


In [ ]:
df_f['developers'] = df_f['developers'].fillna('unknown')
df_f['publishers'] = df_f['publishers'].fillna('unknown')

print('결측 처리 완료')
print(f'developers 결측: {df_f["developers"].isna().sum()}')
print(f'publishers 결측: {df_f["publishers"].isna().sum()}')

### 데이터 저장

전처리된 메인 분석 모집단을 `data/preprocessed/steam_indie_games.csv`로 저장한다.

In [37]:
out_path = PREPROCESSED_DIR / 'steam_indie_games.csv'
df_f.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_f):,}개)')
print(f'컬럼: {df_f.columns.tolist()}')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_games.csv (9,692개)
컬럼: ['appid', 'positive', 'negative', 'price', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher', 'tags']


---

## steam_indie_review_histogram 전처리

`data/raw/steam_indie_review_histogram.csv`를 로드하여 분석에 필요한 파생 컬럼을 추가한다.

| 처리 항목 | 결과 |
|---|---|
| 문자열 공백 | 문자열 컬럼 앞뒤 공백 제거, `data_type` 소문자 통일 |
| 날짜 변환 | `release_date`, `hist_start_date`, `hist_end_date`, `date` → datetime |
| 리뷰 수 합계 | `recommendations_total` 생성 (`recommendations_up + recommendations_down`) |

In [38]:
df_hist = pd.read_csv(RAW_DIR / 'steam_indie_review_histogram.csv')
print(f'로드 완료: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

로드 완료: (11782, 10)
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']


In [39]:
# 문자열 공백 정리
string_cols = df_hist.select_dtypes(include=['object', 'string']).columns.tolist()
for col in string_cols:
    df_hist[col] = df_hist[col].astype('string').str.strip()
if 'name' in df_hist.columns:
    df_hist['name'] = df_hist['name'].str.replace(r'\s+', ' ', regex=True)
df_hist['data_type'] = df_hist['data_type'].str.lower()

# 날짜형 변환
for col in ['release_date', 'hist_start_date', 'hist_end_date', 'date']:
    df_hist[col] = pd.to_datetime(df_hist[col], errors='coerce')

# 리뷰 수 합계
for col in ['appid', 'recommendations_up', 'recommendations_down']:
    df_hist[col] = pd.to_numeric(df_hist[col], errors='coerce')
df_hist['recommendations_total'] = df_hist['recommendations_up'] + df_hist['recommendations_down']

print('전처리 완료')
print(f'shape: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

전처리 완료
shape: (11782, 11)
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type', 'recommendations_total']


### 데이터 저장

전처리 결과를 `data/preprocessed/steam_indie_review_histogram.csv`로 저장한다.

In [40]:
out_path = PREPROCESSED_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_hist):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_review_histogram.csv (11,782개)


---

## steam_indie_reviews 전처리

`data/raw/steam_indie_reviews.csv`를 로드하여 분석에 필요한 형태로 정제한다.

| 처리 항목 | 결과 |
|---|---|
| 결측 제거 | `review` 컬럼 결측 행 제거 |
| 분석 대상 필터링 | `steam_indie_games` 기준 inner join (분석 대상 게임 리뷰만 유지) |
| 날짜 변환 | `timestamp_created` → `created_date`, `timestamp_updated` → `updated_date`, `author_last_played` → `author_last_played_date` |
| 플레이타임 | `author_playtime_*` 3개 컬럼 분 → 시간 단위 변환 후 원본 제거 |

In [41]:
reviews = pd.read_csv(RAW_DIR / 'steam_indie_reviews.csv')
print(f'로드 완료: {reviews.shape}')

# 결측 제거
reviews_clean = reviews.copy()
reviews_clean = reviews_clean.dropna(subset=['review'])

# steam_indie_games 기준 inner join (분석 대상 게임의 리뷰만 유지)
games_appids = pd.read_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', usecols=['appid'])
before = len(reviews_clean)
reviews_clean = reviews_clean.merge(games_appids, on='appid', how='inner')
print(f'inner join 결과: {before:,} → {len(reviews_clean):,}행 ({before - len(reviews_clean):,}개 제외)')

# 날짜 변환 (Unix timestamp → datetime)
for col in ['timestamp_created', 'timestamp_updated', 'author_last_played']:
    new_col = col.replace('timestamp_', '') + '_date' if col.startswith('timestamp_') else 'author_last_played_date'
    reviews_clean[new_col] = pd.to_datetime(reviews_clean[col], unit='s', errors='coerce')

# 플레이타임 분 → 시간 변환 후 원본 제거
playtime_cols = ['author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review']
for col in playtime_cols:
    new_col = col.replace('author_playtime_', 'playtime_') + '_hours'
    reviews_clean[new_col] = reviews_clean[col] / 60
reviews_clean = reviews_clean.drop(columns=playtime_cols)

print(f'전처리 완료: {reviews_clean.shape}')
print(f'컬럼: {reviews_clean.columns.tolist()}')

로드 완료: (236379, 21)
inner join 결과: 235,853 → 235,853행 (0개 제외)
전처리 완료: (235853, 24)
컬럼: ['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_last_played', 'created_date', 'updated_date', 'author_last_played_date', 'playtime_forever_hours', 'playtime_last_two_weeks_hours', 'playtime_at_review_hours']


### 데이터 저장

전처리 결과를 `data/preprocessed/steam_indie_reviews.csv`로 저장한다.

In [42]:
out_path = PREPROCESSED_DIR / 'steam_indie_reviews.csv'
reviews_clean.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(reviews_clean):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_reviews.csv (235,853개)
